# Multi-Sensor Anomaly Detection with MOMENT

This notebook demonstrates **multi-sensor fusion** for anomaly detection in
manufacturing processes.  We use [MOMENT](https://github.com/moment-timeseries-foundation-model/moment),
a time-series foundation model, to detect anomalies across three correlated
sensor channels — temperature, pressure, and vibration — then fuse the
per-channel scores into a single process health assessment.

**What you'll learn:**
1. How to generate realistic multi-channel sensor data with the built-in
   simulator profiles.
2. How to run MOMENT anomaly detection on individual sensor channels.
3. How to combine per-channel scores using the `SensorFusionEngine`.
4. How to visualise normal operation, degradation, and anomaly events.

**Prerequisites:**
```bash
pip install momentfm matplotlib numpy
```

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np

# Make project modules importable
REPO_ROOT = Path.cwd()
# Adjust if running from the notebooks/ directory
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src" / "sensor-simulator" / "profiles"))
sys.path.insert(0, str(REPO_ROOT / "samples" / "03-end-to-end"))

from temperature_furnace import TemperatureFurnaceProfile
from pressure_hydraulic import PressureHydraulicProfile
from multi_sensor_fusion import SensorFusionEngine

# Optional: MOMENT model
try:
    import torch
    from momentfm import MOMENTPipeline
    HAS_MOMENT = True
except ImportError:
    HAS_MOMENT = False
    print("momentfm not installed — using heuristic scoring. Install with: pip install momentfm")

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline

SAMPLING_RATE = 10.0
WINDOW_SIZE = 512

print(f"MOMENT available: {HAS_MOMENT}")
print(f"Window size: {WINDOW_SIZE} samples ({WINDOW_SIZE / SAMPLING_RATE:.0f}s at {SAMPLING_RATE} Hz)")

## Generate Sensor Data

We create three sensor profiles that simulate a manufacturing process:

| Channel      | Profile                      | Normal range       |
|-------------|------------------------------|--------------------|
| Temperature | Industrial furnace (°C)      | 25 – 650 °C       |
| Pressure    | Hydraulic system (bar)       | ~150 bar setpoint  |
| Vibration   | Motor bearing (g RMS)        | ~2.5 g baseline    |

We generate **6 windows** of data, injecting faults at windows 3 and 5.

In [ ]:
# --- Vibration helper (lightweight, no separate profile module yet) ---
class VibrationProfile:
    def __init__(self, sampling_rate=10.0, base_rms=2.5, noise_level=0.3):
        self.sampling_rate = sampling_rate
        self.base_rms = base_rms
        self.noise_level = noise_level
        self._rng = np.random.default_rng(42)
        self._t = 0.0
        self._fault = False

    def generate(self, n):
        dt = 1.0 / self.sampling_rate
        t = self._t + np.arange(n) * dt
        self._t = t[-1] + dt
        sig = (
            self.base_rms * np.sin(2 * np.pi * 1.0 * t)
            + 0.4 * self.base_rms * np.sin(2 * np.pi * 3.0 * t)
            + 0.15 * self.base_rms * np.sin(2 * np.pi * 7.0 * t)
        )
        if self._fault:
            sig += 3.0 * self.base_rms * np.abs(np.sin(2 * np.pi * 0.4 * t)) ** 8
        return sig + self._rng.normal(0, self.noise_level, n)

    def inject_anomaly(self, _type="bearing_fault"):
        self._fault = True


# --- Create profiles ---
temp_profile = TemperatureFurnaceProfile(sampling_rate=SAMPLING_RATE, noise_level=1.5)
pres_profile = PressureHydraulicProfile(sampling_rate=SAMPLING_RATE, noise_level=0.8)
vib_profile  = VibrationProfile(sampling_rate=SAMPLING_RATE)

N_WINDOWS = 6
data = {"temperature": [], "pressure": [], "vibration": []}
phase_names = [
    "Normal", "Normal",
    "Drift + Leak", "Degradation",
    "Acute fault", "Recovery",
]

for w in range(N_WINDOWS):
    if w == 2:
        temp_profile.inject_anomaly("calibration_drift")
        pres_profile.inject_anomaly("leak")
    if w == 4:
        temp_profile.inject_anomaly("thermal_shock")
        pres_profile.inject_anomaly("cavitation")
        vib_profile.inject_anomaly("bearing_fault")

    data["temperature"].append(temp_profile.generate(WINDOW_SIZE))
    data["pressure"].append(pres_profile.generate(WINDOW_SIZE))
    data["vibration"].append(vib_profile.generate(WINDOW_SIZE))
    print(f"Window {w+1}: {phase_names[w]:20s}  "
          f"temp=[{data['temperature'][-1].min():.0f}, {data['temperature'][-1].max():.0f}] °C   "
          f"pres=[{data['pressure'][-1].min():.0f}, {data['pressure'][-1].max():.0f}] bar   "
          f"vib=[{data['vibration'][-1].min():.1f}, {data['vibration'][-1].max():.1f}] g")

## Individual Channel Anomaly Detection

We run MOMENT's reconstruction task on each window of each channel.  The
anomaly score is derived from the mean reconstruction error — higher error
indicates the model found the pattern unusual.

In [ ]:
# --- Load MOMENT or fall back to heuristic ---
model = None
if HAS_MOMENT:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = MOMENTPipeline.from_pretrained(
            "AutonLab/MOMENT-1-large",
            model_kwargs={"task_name": "reconstruction"},
        )
        model.init()
    print("MOMENT model loaded.")


def anomaly_score(window):
    """Compute anomaly score for a single window."""
    if model is not None:
        tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        with torch.no_grad():
            out = model(tensor)
        recon = out.reconstruction.squeeze().numpy()
        mae = float(np.mean(np.abs(window - recon)))
        return round(1.0 / (1.0 + np.exp(-2.0 * (mae - 1.5))), 4)
    else:
        # Heuristic: z-score of tail vs. full window
        mean, std = np.mean(window), np.std(window) + 1e-8
        z = np.mean(np.abs((window[-64:] - mean) / std))
        return round(float(1.0 / (1.0 + np.exp(-1.5 * (z - 2.0)))), 4)


# --- Score every channel × window ---
scores = {ch: [] for ch in data}
for ch in data:
    for w_idx, window in enumerate(data[ch]):
        s = anomaly_score(window)
        scores[ch].append(s)

# Display results
print(f"{'Window':<8} {'Temperature':>12} {'Pressure':>10} {'Vibration':>10}")
print("-" * 44)
for w in range(N_WINDOWS):
    print(f"{w+1:<8} {scores['temperature'][w]:>12.4f} {scores['pressure'][w]:>10.4f} {scores['vibration'][w]:>10.4f}")

## Sensor Fusion

Individual channel scores may fluctuate.  The `SensorFusionEngine`
combines them into a single **fused anomaly score** using a weighted
average.  Temperature and pressure get higher weights because they are
more process-critical in this scenario.

We also demonstrate switching between fusion strategies.

In [ ]:
# --- Set up the fusion engine ---
engine = SensorFusionEngine(strategy="weighted_average", global_threshold=0.7)
engine.add_channel("temperature", weight=1.2, threshold=0.65)
engine.add_channel("pressure",    weight=1.0, threshold=0.70)
engine.add_channel("vibration",   weight=0.8, threshold=0.60)

# --- Feed scores window by window ---
fused_scores = []
anomaly_flags = []

print(f"{'Window':<8} {'Fused':>7} {'Anomaly?':>10}  Details")
print("-" * 65)

for w in range(N_WINDOWS):
    engine.update("temperature", scores["temperature"][w])
    engine.update("pressure",    scores["pressure"][w])
    engine.update("vibration",   scores["vibration"][w])

    fused = engine.fused_score()
    is_anom = engine.is_anomaly()
    fused_scores.append(fused)
    anomaly_flags.append(is_anom)

    status = engine.get_status()
    triggered = [c["name"] for c in status["channels"] if c["is_anomaly"]]
    detail = ", ".join(triggered) if triggered else "all normal"
    flag = "🚨 YES" if is_anom else "✅ No"
    print(f"{w+1:<8} {fused:>7.4f} {flag:>10}  [{detail}]")

# --- Compare strategies ---
print("\n--- Strategy comparison (last window) ---")
for strat in ["weighted_average", "max", "voting"]:
    engine.strategy = strat
    print(f"  {strat:>20s}: fused={engine.fused_score():.4f}  anomaly={engine.is_anomaly()}")
engine.strategy = "weighted_average"  # reset

## Visualization

The plot below shows:
- **Top 3 panels**: Raw sensor data for each channel.  Red-shaded regions
  mark windows flagged as anomalous by the fusion engine.
- **Bottom panel**: Fused anomaly score per window.  The dashed red line
  is the alert threshold.

In [ ]:
channels = ["temperature", "pressure", "vibration"]
colours  = {"temperature": "#e74c3c", "pressure": "#3498db", "vibration": "#2ecc71"}
units    = {"temperature": "°C", "pressure": "bar", "vibration": "g (RMS)"}

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for idx, ch in enumerate(channels):
    ax = axes[idx]
    full = np.concatenate(data[ch])
    t = np.arange(len(full)) / SAMPLING_RATE
    ax.plot(t, full, color=colours[ch], linewidth=0.6, alpha=0.9)
    ax.set_ylabel(f"{ch}\n({units[ch]})", fontsize=9)
    ax.grid(True, alpha=0.3)
    # Shade anomaly windows
    for w_idx, is_a in enumerate(anomaly_flags):
        if is_a:
            t0 = w_idx * WINDOW_SIZE / SAMPLING_RATE
            t1 = (w_idx + 1) * WINDOW_SIZE / SAMPLING_RATE
            ax.axvspan(t0, t1, color="red", alpha=0.12)

# Fused score bar chart
ax_s = axes[-1]
centres = [(i + 0.5) * WINDOW_SIZE / SAMPLING_RATE for i in range(N_WINDOWS)]
bar_c = ["#e74c3c" if f else "#2ecc71" for f in anomaly_flags]
ax_s.bar(centres, fused_scores, width=WINDOW_SIZE / SAMPLING_RATE * 0.8,
         color=bar_c, alpha=0.8, edgecolor="white")
ax_s.axhline(0.7, color="red", linestyle="--", linewidth=1, label="Threshold")
ax_s.set_ylabel("Fused\nAnomaly Score", fontsize=9)
ax_s.set_xlabel("Time (s)")
ax_s.set_ylim(0, 1.05)
ax_s.legend(loc="upper left", fontsize=8)
ax_s.grid(True, alpha=0.3)

fig.suptitle("Multi-Sensor Anomaly Detection with MOMENT", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Conclusion

### Key Takeaways

1. **Multi-sensor fusion catches what individual channels miss.**  A slight
   temperature drift combined with a slow pressure leak may not trip either
   channel's threshold alone, but the fused score highlights the combined
   deviation.

2. **Different fusion strategies suit different use cases:**
   - `weighted_average` — balanced, good default for process monitoring
   - `max` — conservative, triggers on any single-channel anomaly
   - `voting` — consensus-based, reduces false positives

3. **MOMENT provides a strong anomaly detector** out of the box.  The
   reconstruction-based approach works well even without task-specific
   fine-tuning, because the model has learned general time-series patterns
   from large-scale pre-training.

### Next Steps

- Deploy with MQTT: connect the fusion engine to the inference server's
  `MQTTBridge` to process live sensor data.
- Configure AIO dataflows to route anomaly alerts to Azure Event Hubs or
  Azure Data Explorer (see `docs/use-cases/anomaly-detection.md`).
- Tune per-channel thresholds using historical production data.
- Add more channels (flow rate, current draw, acoustic emission).